# EXP-011: False-alarm-rate-matched comparison of Arm C (reactive) vs Arm D (anticipatory) (R8)

**Status: engineering validation / proof of concept**, same caveats as all prior
notebooks this session.

**Purpose (R8 of the revision roadmap, Devil's Advocate/Domain, Critical):** every
prior lead-time comparison (EXP-001, EXP-009) used the SAME absolute threshold for
both Arm C (triggers on current loss) and Arm D (triggers on a 15-step-ahead linear
extrapolation of loss). But a noisier signal compared against the same fixed
threshold as a less-noisy signal will cross it earlier and more often BY
CONSTRUCTION, regardless of whether the extra "earliness" reflects genuine
predictive value or just noise-driven false alarms. Two reviewers independently
flagged that the ~3-7 step lead-time advantage measured so far is consistent with
this artifact and has never been checked against each arm's OWN false-alarm rate.

**This notebook does three things EXP-001/EXP-009 never did:**
1. Adds a genuine **no-disturbance control** condition (nothing ever changes) and
   measures each arm's spontaneous false-trigger rate over a long post-calibration
   window.
2. Calibrates each arm's threshold from **its own decision signal's** own
   calibration-window statistics (current loss for C, forecast value for D) rather
   than reusing a shared loss-based threshold for both -- this alone removes one
   source of the apples-to-oranges comparison.
3. **Matches the two arms' false-alarm rates** (via a small per-arm sensitivity
   sweep against the no-disturbance control) before re-measuring lead time on
   drift/instability under the FAR-matched thresholds -- the comparison R8 actually
   asks for.


In [1]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt

torch.set_num_threads(1)
print("torch", torch.__version__, "| numpy", np.__version__)


C:\Users\USER\anaconda3\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


torch 2.5.1+cpu | numpy 2.0.2


## 1. Toy task, disturbances (+ genuine no-disturbance control), forecaster

In [2]:
def make_model(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))

MEAN0_BASE = np.array([-1.0, -1.0])
MEAN1_BASE = np.array([1.0, 1.0])
BASE_NOISE_SCALE = 0.5

def gen_batch(rng, drift_frac=0.0, batch_size=16):
    frac = float(np.clip(drift_frac, 0.0, 1.0))
    mean0 = (1 - frac) * MEAN0_BASE + frac * MEAN1_BASE
    mean1 = (1 - frac) * MEAN1_BASE + frac * MEAN0_BASE
    y = rng.integers(0, 2, size=batch_size)
    means = np.where(y[:, None] == 0, mean0, mean1)
    x = means + rng.normal(scale=BASE_NOISE_SCALE, size=(batch_size, 2))
    return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

def grad_norm(model):
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += float((p.grad.detach() ** 2).sum())
    return total ** 0.5

N_STEPS = 400
DISTURBANCE_ONSET = 200
RAMP_LEN = 40
GRAD_NOISE_MAX = 8.0
TREND_WINDOW = 5
FORECAST_HORIZON = 15
CALIB_START, CALIB_END = 150, 200   # R3-corrected post-convergence calibration window

def _ramp(step):
    if step < DISTURBANCE_ONSET:
        return 0.0
    return min(step - DISTURBANCE_ONSET, RAMP_LEN) / RAMP_LEN

def _instant(step):
    return 1.0 if step >= DISTURBANCE_ONSET else 0.0

def drift_forecastable(step):
    return (_ramp(step), 0.0)

def instability_forecastable(step):
    return (0.0, GRAD_NOISE_MAX * _ramp(step))

def no_disturbance(step):
    '''Genuine control: nothing ever changes, at any step. Used to measure each
    arm's spontaneous false-trigger rate.'''
    return (0.0, 0.0)

def trend_forecast(history, horizon=1):
    n = len(history)
    if n < 2:
        return history[-1] if n else 0.0
    x = np.arange(n)
    y = np.array(history)
    slope, intercept = np.polyfit(x, y, 1)
    return float(slope * (n - 1 + horizon) + intercept)


## 2. Single training run, EACH ARM CALIBRATED ON ITS OWN DECISION SIGNAL

In [3]:
def run_trial(seed, arm, disturbance_fn, sensitivity_k, n_steps=N_STEPS):
    '''R8 FIX: Arm C's threshold is calibrated from the DISTRIBUTION OF RAW LOSS during
    the calibration window (as before). Arm D's threshold is calibrated from the
    DISTRIBUTION OF THE FORECAST SIGNAL ITSELF during the calibration window (the
    15-step-ahead extrapolation of loss, computed at each calibration step) -- NOT
    from raw loss. This means each arm's threshold reflects the actual noise level of
    the signal it is being compared against, which is the minimum requirement for a
    fair comparison. `sensitivity_k` is the z-multiplier (mean + k*std); swept
    per-arm in section 3 to find a matched false-alarm rate.'''
    assert arm in ("C", "D")
    rng = np.random.default_rng(seed)
    model = make_model(seed)
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = nn.BCEWithLogitsLoss()

    loss_hist = []
    threshold = None
    trigger_step = None

    for step in range(n_steps):
        drift_frac, grad_noise_extra = disturbance_fn(step)
        xb, yb = gen_batch(rng, drift_frac=drift_frac)

        opt.zero_grad()
        hidden = model[1](model[0](xb))
        logits = model[2](hidden).squeeze(-1)
        loss = loss_fn(logits, yb)
        loss.backward()

        loss_t = float(loss.item())
        if grad_noise_extra > 0:
            with torch.no_grad():
                for p in model.parameters():
                    if p.grad is not None:
                        p.grad.add_(torch.randn_like(p.grad) * grad_noise_extra)

        loss_hist.append(loss_t)

        if step == CALIB_END - 1:
            if arm == "C":
                calib_signal = np.array(loss_hist[CALIB_START:CALIB_END])
            else:
                # Arm D's OWN signal: the forecast value computed at each calibration
                # step, using the same trailing-window/horizon it uses at decision time.
                calib_signal = np.array([
                    trend_forecast(loss_hist[max(0, s - TREND_WINDOW + 1):s + 1], horizon=FORECAST_HORIZON)
                    for s in range(CALIB_START, CALIB_END)
                ])
            threshold = float(calib_signal.mean() + sensitivity_k * calib_signal.std() + 1e-6)

        if step >= CALIB_END and trigger_step is None and threshold is not None:
            if arm == "C":
                signal = loss_t
            else:
                window = loss_hist[max(0, step - TREND_WINDOW + 1):step + 1]
                signal = trend_forecast(window, horizon=FORECAST_HORIZON)
            if signal > threshold:
                trigger_step = step

        opt.step()

    return trigger_step


## 3. Per-arm sensitivity sweep against the NO-DISTURBANCE control -- find matched false-alarm rates

In [4]:
FAR_SEEDS = list(range(200, 250))  # 50 fresh seeds, never used elsewhere, control-only
# R3's finding (a 3-sigma/50-sample threshold checked over ~200 repeated steps gives a
# near-total false-alarm rate from multiple-testing compounding) means k needs to go
# much higher than a naive "3 looks strict" intuition to reach a low target FAR here --
# grid extended accordingly rather than assumed.
SENSITIVITY_GRID = [2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 10.0, 12.0, 15.0]

far_rows = []
for arm in ("C", "D"):
    for k in SENSITIVITY_GRID:
        n_false_alarms = sum(
            run_trial(seed, arm, no_disturbance, sensitivity_k=k) is not None
            for seed in FAR_SEEDS
        )
        far = n_false_alarms / len(FAR_SEEDS)
        far_rows.append({"arm": arm, "sensitivity_k": k, "false_alarm_rate": far})

far_df = pd.DataFrame(far_rows)
far_pivot = far_df.pivot(index="sensitivity_k", columns="arm", values="false_alarm_rate")
print(f"False-alarm rate under NO-DISTURBANCE control, {len(FAR_SEEDS)} seeds, by sensitivity_k:")
far_pivot


False-alarm rate under NO-DISTURBANCE control, 50 seeds, by sensitivity_k:


arm,C,D
sensitivity_k,,
2.0,0.96,0.90
3.0,0.88,0.80
4.0,0.86,0.66
5.0,0.72,0.58
6.0,0.64,0.46
7.0,0.50,0.26
8.0,0.46,0.18
10.0,0.24,0.08
12.0,0.08,0.04


In [5]:
TARGET_FAR = 0.10  # find, per arm, the smallest k in the grid whose FAR <= this target

def find_matched_k(arm, target_far=TARGET_FAR):
    sub = far_pivot[arm]
    candidates = sub[sub <= target_far]
    if len(candidates) == 0:
        return float(sub.index.max())  # even the strictest tested k doesn't reach target; use strictest
    return float(candidates.index.min())

k_C = find_matched_k("C")
k_D = find_matched_k("D")
print(f"Matched-FAR sensitivity_k (target FAR <= {TARGET_FAR*100:.0f}%): arm C -> k={k_C}, arm D -> k={k_D}")
print(f"  arm C actual FAR at k={k_C}: {far_pivot.loc[k_C, 'C']*100:.0f}%")
print(f"  arm D actual FAR at k={k_D}: {far_pivot.loc[k_D, 'D']*100:.0f}%")
print()
print("For reference, the ORIGINAL shared threshold (both arms, k=3, loss-based-only for both):")
orig_C_far = far_pivot.loc[3.0, "C"] if 3.0 in far_pivot.index else float("nan")
orig_D_shared = sum(
    run_trial(seed, "D", no_disturbance, sensitivity_k=3.0) is not None
    for seed in FAR_SEEDS
) / len(FAR_SEEDS)
print(f"  arm C FAR at shared k=3: {orig_C_far*100:.0f}%  (own-signal calibration, same as before since C always used loss)")
print(f"  arm D FAR at k=3 using arm D's OWN forecast-signal calibration: {far_pivot.loc[3.0,'D']*100:.0f}%")


Matched-FAR sensitivity_k (target FAR <= 10%): arm C -> k=12.0, arm D -> k=10.0
  arm C actual FAR at k=12.0: 8%
  arm D actual FAR at k=10.0: 8%

For reference, the ORIGINAL shared threshold (both arms, k=3, loss-based-only for both):


  arm C FAR at shared k=3: 88%  (own-signal calibration, same as before since C always used loss)
  arm D FAR at k=3 using arm D's OWN forecast-signal calibration: 80%


## 4. Lead-time comparison at the FAR-MATCHED thresholds

In [6]:
LEAD_SEEDS = list(range(10))  # same seeds as EXP-001/EXP-009 for direct comparability

lead_rows = []
for cond_name, fn in (("drift_forecastable", drift_forecastable), ("instability_forecastable", instability_forecastable)):
    for arm, k in (("C", k_C), ("D", k_D)):
        for seed in LEAD_SEEDS:
            trigger_step = run_trial(seed, arm, fn, sensitivity_k=k)
            lead_time = (trigger_step - DISTURBANCE_ONSET) if trigger_step is not None else None
            lead_rows.append({"condition": cond_name, "arm": arm, "seed": seed,
                               "sensitivity_k": k, "trigger_step": trigger_step, "lead_time": lead_time})

lead_df = pd.DataFrame(lead_rows)
lead_summary = lead_df.groupby(["condition", "arm"]).agg(
    n_triggered=("trigger_step", lambda s: s.notna().sum()),
    mean_lead_time=("lead_time", "mean"),
).round(2)
lead_summary


n_triggered  mean_lead_time
condition                arm                             
drift_forecastable       C             10            11.3
                         D             10            12.6
instability_forecastable C             10            16.1
                         D             10            17.5

## 5. Interpretation

In [7]:
def get_lead(cond, arm):
    return lead_summary.loc[(cond, arm), "mean_lead_time"]

original_diff = {"drift_forecastable": 3.10, "instability_forecastable": 7.90}  # from EXP-009, shared k=3 threshold

interpretation_lines = []
interpretation_lines.append("R8 -- DOES ARM D STILL BEAT ARM C ON LEAD TIME ONCE FALSE-ALARM RATES ARE MATCHED?")
interpretation_lines.append("")
interpretation_lines.append(f"Matched-FAR sensitivity: arm C k={k_C} (FAR={far_pivot.loc[k_C,'C']*100:.0f}%), "
                             f"arm D k={k_D} (FAR={far_pivot.loc[k_D,'D']*100:.0f}%), target <= {TARGET_FAR*100:.0f}%.")
interpretation_lines.append("")
for cond in ("drift_forecastable", "instability_forecastable"):
    new_diff = get_lead(cond, "C") - get_lead(cond, "D")
    orig = original_diff[cond]
    interpretation_lines.append(
        f"{cond}: lead-time diff (C-D), FAR-matched = {new_diff:.2f} steps "
        f"(was {orig:.2f} steps under the OLD shared/unmatched threshold)"
    )

matched_advantage_survives = all(
    (get_lead(cond, "C") - get_lead(cond, "D")) > 0 for cond in ("drift_forecastable", "instability_forecastable")
)

interpretation = chr(10).join(interpretation_lines) + f'''

VERDICT: {"the anticipatory (Arm D) lead-time advantage SURVIVES false-alarm-rate matching -- Arm D still triggers earlier than Arm C on both conditions even after removing the confound of comparing a noisier signal against the same absolute threshold as a less-noisy one. This is meaningfully stronger evidence for the core Stage A novelty claim than the original unmatched comparison, because the two arms are now being held to the same standard of 'how often do you cry wolf' before being compared on 'how fast do you catch the real thing.'" if matched_advantage_survives else "the anticipatory (Arm D) lead-time advantage DOES NOT fully survive false-alarm-rate matching on at least one condition -- once both arms are required to have comparable false-alarm rates under the no-disturbance control, at least part of the originally measured lead-time advantage disappears or reverses. This means the original ~3-7 step advantage was, at least in part, exactly the construction artifact the reviewers predicted: a noisier forecast signal crossing the same threshold earlier by virtue of its own noise, not because it was genuinely anticipating the disturbance. This substantially narrows what can be claimed for the core Stage A novelty claim and should be treated as the current best answer, not the earlier unmatched comparison."}

CAVEATS: (1) the false-alarm-rate sweep used a coarse grid (9 points) and 50 control
seeds -- the matched-k values are approximate, not exact, and a finer sweep or more
control seeds could shift them slightly. (2) TARGET_FAR=10% was chosen as a round,
reasonable number, not derived from any principled deployment requirement -- a
different target FAR could change which arm looks more favorable, since the two
arms' FAR-vs-k curves are not necessarily parallel. (3) this test used only the
`reduce` action's threshold-crossing logic (trigger timing only, not the
downstream intervention-type diagnosis) -- it isolates the TIMING claim specifically,
which is what R8 was scoped to address. (4) same Stage A caveats as all prior
notebooks this session (small seed counts, no pre-registered effect sizes, not the
frozen Stage D experiment).
'''
print(interpretation)


R8 -- DOES ARM D STILL BEAT ARM C ON LEAD TIME ONCE FALSE-ALARM RATES ARE MATCHED?

Matched-FAR sensitivity: arm C k=12.0 (FAR=8%), arm D k=10.0 (FAR=8%), target <= 10%.

drift_forecastable: lead-time diff (C-D), FAR-matched = -1.30 steps (was 3.10 steps under the OLD shared/unmatched threshold)
instability_forecastable: lead-time diff (C-D), FAR-matched = -1.40 steps (was 7.90 steps under the OLD shared/unmatched threshold)

VERDICT: the anticipatory (Arm D) lead-time advantage DOES NOT fully survive false-alarm-rate matching on at least one condition -- once both arms are required to have comparable false-alarm rates under the no-disturbance control, at least part of the originally measured lead-time advantage disappears or reverses. This means the original ~3-7 step advantage was, at least in part, exactly the construction artifact the reviewers predicted: a noisier forecast signal crossing the same threshold earlier by virtue of its own noise, not because it was genuinely anticipat